In [5]:
import pandas as pd

splits = {'train': 'tb_ssa_extra_large_10000.csv', 'validation': 'tb_ssa_large_5000.csv', 'test': 'tb_ssa_test_2000.csv'}
df = pd.read_csv("hf://datasets/electricsheepafrica/africa-synth-tuberculosis-tuberculosis-dataset-all/" + splits["train"])

In [6]:
df.head()

,patient_id,age,age_category,sex,residence,overcrowding,bmi,underweight,hiv_status,cd4_count,...,esr_mm_hr,hemoglobin_g_dl,anemia,xray_abnormal,xray_findings,treatment_started,treatment_category,treatment_outcome,died,tb_probability_score
0,TB000001,12,0-14,Male,Urban,False,14.5,True,Positive,222.0,...,56,11.1,True,True,Upper lobe infiltrates,True,New case - First line,Cured/Completed,False,0.85
1,TB000002,37,50+,Male,Rural,True,14.1,True,Positive,235.0,...,117,14.1,False,True,Upper lobe infiltrates,False,NaN,Not started,False,0.85
2,TB000003,33,25-34,Female,Urban,False,29.0,False,Negative,NaN,...,9,15.9,False,NaN,Normal,NaN,NaN,NaN,False,0.39
3,TB000004,66,50+,Female,Urban,True,24.6,False,Positive,272.0,...,20,12.9,False,True,Consolidation,False,NaN,Not started,False,0.85
4,TB000005,34,35-49,Male,Urban,False,18.5,False,Negative,NaN,...,34,11.5,True,True,Consolidation,True,Retreatment/MDR,Cured/Completed,False,0.85


In [7]:
# Shuffle and split into train, validation, and test sets
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

n_rows = len(df)
train_end = int(n_rows * 0.7)
val_end = int(n_rows * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print(f"Train shape: {train_df.shape}")
print(f"Validation shape: {val_df.shape}")
print(f"Test shape: {test_df.shape}")

Train shape: (7000, 36)
Validation shape: (1500, 36)
Test shape: (1500, 36)


In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Use the tuberculosis probability score as the regression target
target_col = "tb_probability_score" if "tb_probability_score" in df.columns else None
if target_col is None:
    raise ValueError("Could not find a tuberculosis probability column. Please choose a numeric target manually.")

print(f"Using target column: {target_col}")
feature_cols = [col for col in df.columns if col not in {target_col, "patient_id"}]

X_train = train_df[feature_cols]
y_train = train_df[target_col]
X_test = test_df[feature_cols]
y_test = test_df[target_col]

# Build a regression pipeline that handles numeric and categorical features
numeric_features = X_train.select_dtypes(include=["number"]).columns
categorical_features = X_train.select_dtypes(exclude=["number"]).columns

if len(numeric_features) > 0 and len(categorical_features) > 0:
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]
    )
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )
    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
        ]
    )
elif len(numeric_features) > 0:
    model = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("regressor", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
        ]
    )
else:
    model = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
            ("regressor", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
        ]
    )

model.fit(X_train, y_train)

# Create a leaderboard table that appends new model results
leaderboard = pd.DataFrame(
    columns=[
        "model_name",
        "train_mae",
        "train_rmse",
        "train_r2",
        "test_mae",
        "test_rmse",
        "test_r2",
    ]
)


def add_model_to_leaderboard(model_name, model_obj, X_train_df, X_test_df, y_train_true, y_test_true):
    global leaderboard

    train_pred_vals = pd.Series(model_obj.predict(X_train_df))
    test_pred_vals = pd.Series(model_obj.predict(X_test_df))

    row = {
        "model_name": model_name,
        "train_mae": mean_absolute_error(y_train_true, train_pred_vals),
        "train_rmse": mean_squared_error(y_train_true, train_pred_vals) ** 0.5,
        "train_r2": r2_score(y_train_true, train_pred_vals),
        "test_mae": mean_absolute_error(y_test_true, test_pred_vals),
        "test_rmse": mean_squared_error(y_test_true, test_pred_vals) ** 0.5,
        "test_r2": r2_score(y_test_true, test_pred_vals),
    }

    leaderboard = pd.concat([leaderboard, pd.DataFrame([row])], ignore_index=True)
    return leaderboard

leaderboard = add_model_to_leaderboard("Random Forest Regressor", model, X_train, X_test, y_train, y_test)
leaderboard = leaderboard.sort_values("test_r2", ascending=False)
leaderboard

Using target column: tb_status


/var/folders/s0/y_l2y8m93wb_tz9xhjks91t80000gn/T/ipykernel_25706/4060400082.py:134: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  leaderboard = pd.concat([leaderboard, pd.DataFrame([row])], ignore_index=True)


,model_name,train_accuracy,train_precision,train_recall,train_f1,train_roc_auc,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
0,Logistic Regression,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [9]:
# Create a probability score table for the test set
predicted_prob = model.predict_proba(X_test)[:, 1]
positive_label = labels[1] if len(labels) > 1 else labels[0]

prediction_scores = pd.DataFrame({
    "actual_label": y_test.reset_index(drop=True),
    "predicted_probability": predicted_prob,
    "predicted_label": [positive_label if prob >= 0.5 else labels[0] for prob in predicted_prob],
})

prediction_scores.head()

,actual_label,predicted_probability,predicted_label
0,Negative,0.000333,Negative
1,Positive,0.999995,Positive
2,Negative,0.000223,Negative
3,Negative,0.000946,Negative
4,Positive,0.999998,Positive


1) Data analysis and visualisation similar to last time

Data cleaning, summaries, and visualisations (numeric distributions, categorical counts, correlation heatmap).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Work on a copy to keep original raw data intact
df_clean = df.copy()

# Report missing values
missing = df_clean.isna().sum().sort_values(ascending=False)
print("Missing values (columns with any missing):")
print(missing[missing > 0])

# Fill numeric columns with median, categorical with mode
num_cols = df_clean.select_dtypes(include=["number"]).columns.tolist()
cat_cols = df_clean.select_dtypes(exclude=["number"]).columns.tolist()

if len(num_cols) > 0:
    df_clean[num_cols] = df_clean[num_cols].fillna(df_clean[num_cols].median())

for c in cat_cols:
    if df_clean[c].isna().any():
        mode = df_clean[c].mode()
        fill_val = mode.iloc[0] if not mode.empty else "missing"
        df_clean[c] = df_clean[c].fillna(fill_val)

print('\nData summary (numeric):')
print(df_clean[num_cols].describe().T)
print('\nSample of cleaned dataframe:')
display(df_clean.head())

In [ ]:
# Distribution of target (if present) and numeric features
plt.rcParams.update({'figure.max_open_warning': 0})
if 'tb_probability_score' in df_clean.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df_clean['tb_probability_score'], bins=30, kde=True, color='#4C72B0')
    plt.title('tb_probability_score distribution')
    plt.xlabel('tb_probability_score')
    plt.show()

# Numeric histograms (first 12 numeric columns)
if len(num_cols) > 0:
    cols_to_plot = num_cols[:12]
    df_clean[cols_to_plot].hist(figsize=(14, 8), bins=20)
    plt.tight_layout()
    plt.show()

# Top categorical counts (up to first 6 categorical columns)
for c in cat_cols[:6]:
    plt.figure(figsize=(8,4))
    order = df_clean[c].value_counts().index
    sns.countplot(data=df_clean, y=c, order=order, palette='pastel')
    plt.title(f'Value counts: {c}')
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation heatmap for numeric features
if len(num_cols) > 1:
    corr = df_clean[num_cols].corr()
    plt.figure(figsize=(12,10))
    sns.heatmap(corr, cmap='coolwarm', center=0, annot=False)
    plt.title('Correlation matrix (numeric features)')
    plt.tight_layout()
    plt.show()

# Quick pairplot for up to 4 numeric columns (small subset to avoid heavy plotting)
if len(num_cols) >= 2:
    pair_cols = num_cols[:4]
    try:
        sns.pairplot(df_clean[pair_cols].sample(min(len(df_clean), 1000), random_state=42))
        plt.suptitle('Pairplot (sample)', y=1.02)
        plt.show()
    except Exception:
        print('Pairplot skipped due to plotting constraints or too many unique values.')